In [77]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from data_handler import EleHandler
from classifier import Classifier
from backbone import Backbone
from concept_head import ConceptHead
from projector import Projector

from torch.utils.data import DataLoader, Subset
from torch.utils.data.sampler import RandomSampler

import torch, math
import torch.nn as nn
import torch.optim as optim
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Data

In [78]:
dataset = EleHandler(subset='EFA_IDI')

train_indices, val_indices, test_indices = dataset.split_along_encounters([0.7,0.15,0.15])

train_loader, val_loader, test_loader = (DataLoader(dataset, batch_size=128, sampler=RandomSampler(indices), num_workers=4) for indices in [train_indices, val_indices, test_indices])

# Check the values of the labels
max_ele_label = 0
for batch in train_loader:
    max_ele_label = max(torch.max(batch[2]), max_ele_label)

print('Number of elephants:', max_ele_label.item() + 1)

num_classes = max_ele_label.item() + 1
code = 'small_Elephant_Classifier'

Unique groups: 542
Train indices: 6108, Validation indices: 1101, Test indices: 1235
Number of elephants: 1484


### Training stage 1 : Concept Head
- loading a pretrained backbone, freeze it
- Train the concept head to learn the SEEK codes
- Test the concept head


In [79]:

ba = Backbone(pretraining = "savannah_elephants")
ba.freeze()

ch = ConceptHead(print_every=1, experiment_code=code)
#ch.train(train_loader, val_loader, ba, num_epochs=30)
ch.test(test_loader, ba)

ch.freeze()

ConceptHead - Test Loss: 0.0311, Test Accuracy: 93.69%
sex Accuracy: 97.18%
age Accuracy: 97.61%
right_tusk Accuracy: 95.19%
left_tusk Accuracy: 95.81%
R_tear_1 Accuracy: 89.31%
R_hole_1 Accuracy: 90.43%
R_tear_2 Accuracy: 89.09%
R_hole_2 Accuracy: 94.75%
L_tear_1 Accuracy: 86.76%
L_hole_1 Accuracy: 92.72%
L_tear_2 Accuracy: 87.94%
L_hole_2 Accuracy: 94.36%
right_extreme Accuracy: 97.85%
left_extreme Accuracy: 96.78%
ear_special Accuracy: 95.33%
body_special Accuracy: 97.93%
average Accuracy: 93.69%
whole_code Accuracy: 48.30%


### Training stage 2 : Cassifier

- With a frozen backbone, train a classifier to learn the elephant id from the backbone original embeddings
- Freeze the classifier|


In [80]:

cl = Classifier( num_classes=num_classes, experiment_code=code)
cl.train(train_loader, val_loader, backbone = ba, num_epochs=30)
cl.test(test_loader, backbone = ba) 

# Freeze the classifier layer
cl.freeze()

weights not found
Training the classifier : Backbone parameters require gradients: False 
Classifier parameters require gradients: True


/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch 1 - Train Loss: 7.2978, Train Accuracy: 0.77% - Val Loss: 7.2844, Val Accuracy: 2.09%
Epoch 2 - Train Loss: 7.2677, Train Accuracy: 4.37% - Val Loss: 7.2038, Val Accuracy: 11.35%
Epoch 3 - Train Loss: 7.1964, Train Accuracy: 11.79% - Val Loss: 7.1702, Val Accuracy: 14.44%
Epoch 4 - Train Loss: 7.1639, Train Accuracy: 14.51% - Val Loss: 7.1499, Val Accuracy: 15.35%
Epoch 5 - Train Loss: 7.1443, Train Accuracy: 16.26% - Val Loss: 7.1376, Val Accuracy: 16.71%
Epoch 6 - Train Loss: 7.1304, Train Accuracy: 17.49% - Val Loss: 7.1210, Val Accuracy: 18.62%
Epoch 7 - Train Loss: 7.1172, Train Accuracy: 18.86% - Val Loss: 7.1052, Val Accuracy: 19.80%
Epoch 8 - Train Loss: 7.1087, Train Accuracy: 19.66% - Val Loss: 7.0922, Val Accuracy: 21.25%
Epoch 9 - Train Loss: 7.1017, Train Accuracy: 20.27% - Val Loss: 7.0851, Val Accuracy: 22.07%
Epoch 10 - Train Loss: 7.0939, Train Accuracy: 21.15% - Val Loss: 7.0782, Val Accuracy: 22.62%
Epoch 11 - Train Loss: 7.0872, Train Accuracy: 21.79% - Val Lo

### Training stage 3 : Projector

- With a frozen backbone and the frozen classifier, train the projector
- The projector trains to project the concepts to the embedding space 
- The loss is computed such that edited embeddings (original embeddings + projected embeedings) correctly predict the elephant id using the classifier learnt is stage 2

In [92]:
def no_intervention(predicted_concepts, true_concepts, correction_probability = 0.0):
    return predicted_concepts

def intervene(predicted_concepts, true_concepts, correction_probability = 0.5):
    for i in range(predicted_concepts.shape[0]):
        if torch.rand(1).item() < correction_probability:
            predicted_concepts[i] = true_concepts[i]
    return predicted_concepts

def perfect_correction(predicted_concepts, true_concepts, correction_probability = 1.0):
    return true_concepts.to('cuda')

In [82]:
pr = Projector(experiment_code=code) 

ba.freeze() # freeze the backbone
ch.freeze() # freeze the concept head
cl.freeze() # freeze the classifier
pr.unfreeze() # unfreeze the projector

pr.train(train_loader, val_loader, ba,  ch, cl, num_epochs=30, intervention_fn=no_intervention)
pr.test(test_loader, ba, cl, ch)


training projector-  weights are reset
Concept head parameters require gradients: False 
Classifier parameters require gradients: False 
Backbone parameters require gradients: False 
Projector parameters require gradients: True


Epoch 1/30 - Train Loss: 7.0522, Train Accuracy: 25.21% - Val Loss: 7.0310, Val Accuracy: 27.28%
Epoch 2/30 - Train Loss: 7.0509, Train Accuracy: 25.27% - Val Loss: 7.0322, Val Accuracy: 27.13%
Epoch 3/30 - Train Loss: 7.0508, Train Accuracy: 25.27% - Val Loss: 7.0316, Val Accuracy: 27.19%
Epoch 4/30 - Train Loss: 7.0507, Train Accuracy: 25.30% - Val Loss: 7.0322, Val Accuracy: 27.13%
Epoch 5/30 - Train Loss: 7.0503, Train Accuracy: 25.32% - Val Loss: 7.0322, Val Accuracy: 27.13%
Epoch 6/30 - Train Loss: 7.0502, Train Accuracy: 25.33% - Val Loss: 7.0293, Val Accuracy: 27.42%
Epoch 7/30 - Train Loss: 7.0503, Train Accuracy: 25.35% - Val Loss: 7.0298, Val Accuracy: 27.36%
Epoch 8/30 - Train Loss: 7.0510, Train Accuracy: 25.25% - Val Loss: 7.0309, Val Accuracy: 27.28%
Epoch 9/30 - Train Loss: 7.0497, Train Accuracy: 25.40% - Val Loss: 7.0319, Val Accuracy: 27.16%
Epoch 10/30 - Train Loss: 7.0493, Train Accuracy: 25.42% - Val Loss: 7.0284, Val Accuracy: 27.51%
Epoch 11/30 - Train Loss: 7.0

Even without correcting the concepts, we improve the accuracy. It seems that we are above the interpretability trade-off. As if, by implementing  some ecology-expertise in the model, we can do better that a blackbox MegaDescriptor.



In [83]:
pr.test(test_loader, ba, cl, ch, intervention_fn=intervene)

Test Loss: 7.0285, Test Accuracy: 27.49%


However, when correcting the concepts at inference time, this seems to be reducing accuracy. Thus we will do a 4th training stage where we correct concepts sometimes.

### Training stage 4 : Projector and Classifier jointly under partial intervention

- With the same architecture, unfreeze the classifier and the projector. Keep the concept head and the backbone frozen
- Train jointly the Projector and Classifier to maximize elephant classification
- When training, correct the concepts with probability 0.5

In [84]:
print('intervention at level 0.5')

pr.unfreeze()
cl.unfreeze()
ch.freeze()
ba.freeze()

pr.train(train_loader, val_loader, ba, ch, cl, num_epochs=30, intervention_fn = intervene, reset_wegihts=False) 


intervention at level 0.5
training projector-  weights are not reset
Concept head parameters require gradients: False 
Classifier parameters require gradients: True 
Backbone parameters require gradients: False 
Projector parameters require gradients: True


Epoch 1/30 - Train Loss: 7.0481, Train Accuracy: 25.59% - Val Loss: 7.0303, Val Accuracy: 27.31%
Epoch 2/30 - Train Loss: 7.0480, Train Accuracy: 25.67% - Val Loss: 7.0278, Val Accuracy: 27.57%
Epoch 3/30 - Train Loss: 7.0483, Train Accuracy: 25.53% - Val Loss: 7.0288, Val Accuracy: 27.48%
Epoch 4/30 - Train Loss: 7.0473, Train Accuracy: 25.65% - Val Loss: 7.0251, Val Accuracy: 27.83%
Epoch 5/30 - Train Loss: 7.0460, Train Accuracy: 25.84% - Val Loss: 7.0233, Val Accuracy: 28.00%
Epoch 6/30 - Train Loss: 7.0461, Train Accuracy: 25.78% - Val Loss: 7.0242, Val Accuracy: 27.94%
Epoch 7/30 - Train Loss: 7.0460, Train Accuracy: 25.78% - Val Loss: 7.0235, Val Accuracy: 28.09%
Epoch 8/30 - Train Loss: 7.0454, Train Accuracy: 25.83% - Val Loss: 7.0237, Val Accuracy: 28.00%
Epoch 9/30 - Train Loss: 7.0451, Train Accuracy: 25.85% - Val Loss: 7.0188, Val Accuracy: 28.46%
Epoch 10/30 - Train Loss: 7.0443, Train Accuracy: 25.92% - Val Loss: 7.0206, Val Accuracy: 28.35%
Epoch 11/30 - Train Loss: 7.0

In [ ]:
pr.test(test_loader, ba, cl, ch, intervention_fn = no_intervention)
pr.test(test_loader, ba, cl, ch, intervention_fn = intervene)
pr.test(test_loader, ba, cl, ch, intervention_fn = perfect_correction)

Test Loss: 7.0129, Test Accuracy: 29.08%
Test Loss: 7.0138, Test Accuracy: 28.99%


### Training stage 5 : ArcFace

In [89]:
import torch
import math
import torch.nn.functional as F

class ArcFaceWithMargin(torch.nn.Module):
    def __init__(self, num_classes, s=30.0, margin=0.5):
        """
        ArcFace with Angular Margin, with added numerical stability.
        Args:
            num_classes: Number of classes in the classification task.
            s: Scale factor to amplify logits for softmax.
            margin: Angular margin (in radians) to enhance class separability.
        """
        super(ArcFaceWithMargin, self).__init__()
        self.s = s  # Scale factor
        self.margin = margin  # Angular margin
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)

    def forward(self, logits: torch.Tensor, labels: torch.Tensor):
        """
        Args:
            logits: Raw logits from the linear layer (batch_size, num_classes).
            labels: Ground truth labels (batch_size,).
        Returns:
            loss: Cross-Entropy Loss with Angular Margin.
        """
        # Normalize the logits
        norm = torch.norm(logits, dim=1, keepdim=True) + 1e-8
        logits_norm = logits / norm

        # Create one-hot encoding for the labels
        one_hot = torch.zeros_like(logits).scatter(1, labels.view(-1, 1), 1.0)

        # Compute cos(theta) for the target classes
        cos_theta = logits_norm.gather(1, labels.view(-1, 1)).squeeze(1)
        cos_theta = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7)  # Tighter clamping for stability

        # Calculate sin(theta)
        sin_theta_squared = 1.0 - cos_theta**2
        sin_theta_squared = sin_theta_squared.clamp(0, 1)  # Ensure no negative values
        sin_theta = torch.sqrt(sin_theta_squared)

        # Calculate cos(theta + margin)
        cos_theta_m = cos_theta * self.cos_m - sin_theta * self.sin_m

        # Ensure numerical stability for the margin update
        cos_theta_m = cos_theta_m.clamp(-1, 1)

        # Update logits with the margin
        logits_with_margin = logits_norm.clone()
        logits_with_margin.scatter_(1, labels.view(-1, 1), cos_theta_m.unsqueeze(1))

        # Scale the logits
        scaled_logits = logits_with_margin * self.s

        # Compute Cross-Entropy Loss
        loss = F.cross_entropy(scaled_logits, labels)
        #print('loss', loss)
        return loss


In [87]:
ba.freeze()
ch.freeze()
pr.unfreeze()
cl.unfreeze()

#pr.loss_fn = ArcFace(num_classes)
#pr.optimizer = optim.Adam(pr.layer.parameters(), lr=1e-5)
pr.loss_fn = ArcFaceWithMargin(num_classes)

pr.train(train_loader, val_loader, ba, ch, cl, num_epochs=30, intervention_fn = no_intervention, reset_wegihts=False) 
pr.test(test_loader, ba, cl, ch, intervention_fn = no_intervention)

training projector-  weights are not reset
Concept head parameters require gradients: False 
Classifier parameters require gradients: True 
Backbone parameters require gradients: False 
Projector parameters require gradients: True


loss tensor(33.1984, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(30.1194, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(30.9662, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(31.8579, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(31.4778, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(30.9455, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(33.0476, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(31.6979, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(33.5684, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(30.7508, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(31.5427, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(32.2951, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(28.3576, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(31.2949, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(32.8207, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tenso

In [93]:
pr.test(test_loader, ba, cl, ch, intervention_fn = intervene)
pr.test(test_loader, ba, cl, ch, intervention_fn = perfect_correction)

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


loss tensor(27.8805, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(25.5325, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.0217, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.1380, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(24.5870, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(27.4614, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(27.4284, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.3952, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(25.7046, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.5524, device='cuda:0', grad_fn=<NllLossBackward0>)
Test Loss: 26.3702, Test Accuracy: 28.16%
loss tensor(26.7230, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.6100, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(28.6567, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(26.8139, device='cuda:0', grad_fn=<NllLossBackward0>)
loss tensor(30.3853, device='cuda: